##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 Firebase Genkit、Ollama 和 Gemma 建置 RAG 應用程式



在這個綜合教學中，您將學習如何使用尖端技術建立 **檢索增強生成 (RAG)** 應用程式：
[**Genkit**](https://firebase.google.com/docs/genkit) 是 framework，旨在幫助您建立人工智慧驅動的應用程式和功能。它提供 Node.js 和 Go 的開源 library，以及用於測試和偵錯的開發人員工具。
[**Gemma**](https://ai.google.dev/gemma) 是 Google 推出的一系列輕量級、最先進的開放語言模型。 Gemma 模型採用與創建 Gemini 模型相同的研究和技術構建而成，是文本到文本、僅限解碼器的英語大語言模型 (LLM)，具有開放權重、預訓練變體和指令調整變體。
[**Ollama**](https://ollama.ai/) 是簡化本地執行語言模型的工具。它允許您有效率地管理和服務多個模型，從而更輕鬆地在電腦上部署和測試 AI 模型。使用Ollama，您可以在不同模型和版本之間無縫切換，為開發和實驗提供靈活性。
[**Firebase**](https://firebase.google.com/) 是 Google 的綜合應用程式開發平台，提供即時資料庫、身份驗證、雲端儲存、託管和機器學習等服務。在本教學中，您將利用 **Cloud Firestore**，這是一個可擴展、靈活的 NoSQL 雲端資料庫，用於儲存和同步用戶端和伺服器端開發的資料。
[**Gradio**](https://gradio.app/) 是一個開源 Python library，用於建立使用者友好的 Web 介面以與機器學習模型進行互動。它允許您快速建立可自訂的 UI 元件來與模型交互，還可以產生任何人都可以使用的可共用 Web 應用程式。
透過整合這些技術，您將建立一個強大的 RAG 應用程序，能夠根據您的自訂資料提供準確且與上下文相關的回應。
## 你將學到什麼

- **設定開發環境**：在 Colab notebook 安裝和設定 Node.js、Genkit、Firebase、Ollama 和 Gradio。
- **使用 Dotprompt 管理提示**：使用 **Dotprompt** 將 prompt 模組化為單獨的 `.prompt` 文件，以實現更好的組織和可維護性。
- **使用 Genkit 流程索引文件**：使用 Genkit 流程嵌入和索引您的數據，使其可供您的 RAG 應用程式檢索。
- **建立聊天機器人介面**：使用 Gradio 創建用戶友好的 chatbot 介面以與您的應用程式互動。


讓我們開始建立您的 RAG 應用程式！
<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Using_with_Firebase_Genkit_and_Ollama.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## **設定**

在開始之前，請確保您擁有：
- 一個基本Google Cloud 帳戶。
- Node.js 和 TypeScript 的基本知識。
- 熟悉Colab notebooks。
- 安裝了最新版本的**Google Cloud SDK**。

## 選擇Colab執行時

在本部分中，您將設定 Google Colab 並設定該項目所需的工具。您將使用 Google Colab 作為執行程式碼的環境，因此請務必仔細遵循這些步驟。
1. **開啟Google Colab**並建立新的notebook。
2. 在 Colab 視窗的右上角，按一下 **▾（其他連線選項）** 按鈕。
3. 選擇**更改 runtime 類型**。
4. 在 **硬體加速器** 下，選擇 **GPU**。
5. 確保 **GPU 類型** 設定為 **T4**。

此設定將為您提供足夠的運算能力來順利執行 Gemma 模型。
**完成這些步驟後，您就可以進入下一部分，在 Colab 環境中設定環境變數。 **

### 設定您的憑證

首先，從以下位置取得您的 Google API 金鑰：https://aistudio.google.com/app/apikey
您需要在Google Colab 中設定Google AI Studio 的憑證。這允許您對不同的服務進行身份驗證並安全地交互，例如Google AI Studio。

1. 開啟 Google Colab notebook 並點選左側面板中的 🔑 Secrets 標籤。 <img src="https://storage.googleapis.com/generativeai-downloads/images/secrets.jpg" alt="The Secrets tab is found on the left panel." width=50%>
2. **新增 Google API 金鑰**：
   - 建立一個名為`GOOGLE_API_KEY`的新secret。
   - 將您的 Google API 金鑰貼到值輸入框中。
   - 切換按鈕以允許notebook 存取secret。

## **安裝依賴項**

要建立RAG應用程序，您需要安裝各種工具和庫. 讓我們開始安裝依賴項。

In [ ]:
# Install Gradio
!pip install -q gradio

# Install Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.ai/install.sh | sh

# Install Node.js
!curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash -
!sudo apt-get install -y nodejs

# Install Genkit CLI and plugins
!npm i -g genkit
!npm i --save genkitx-ollama
!npm i --save @genkit-ai/firebase
!npm i --save @genkit-ai/googleai
!npm i --save @genkit-ai/dotprompt
!npm i llm-chunk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 533.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2

## 使用Ollama執行Gemma

您將使用 `Ollama` 在本地執行 Gemma 語言模型。該工具將允許您與 AI 互動並在 RAG chatbot 應用程式中使用它。

首先，啟動Ollama伺服器。這將在後台執行，並允許您呼叫不同的人工智慧模型。

In [ ]:
import subprocess
import time

ollama_serve_process = subprocess.Popen("OLLAMA_KEEP_ALIVE=-1 ollama serve", shell=True)
time.sleep(5)

Ollama提供library預先設定模型，包含Gemma 2個模型。您可以在[Ollama Gemma 2 型號目錄](https://ollama.com/library/gemma2) 瀏覽可用的Gemma 2 型號。這使您可以輕鬆地在不同Gemma 2 型號之間切換。在此 notebook 中，您將使用 [gemma2:2b](https://ollama.com/library/gemma2:2b) 模型。
要測試 Gemma 模型是否正確執行，請使用以下命令向模型詢問一個簡單的問題：

In [ ]:
ollama_run_process = subprocess.Popen(
  "ollama run gemma2:2b 'What is the capital of China?'",
  shell=True, stdout=subprocess.PIPE, text=True
)

print(ollama_run_process.communicate()[0])

The capital of China is **Beijing**. 





您應該在輸出中看到模型的響應。

##  設定 Firebase 項目

Firebase 將用於儲存和管理您的資料。在本例中，您將使用 Cloud Firestore，這是一個 NoSQL 資料庫，可以輕鬆儲存和檢索 chatbot 將使用的資訊。
在繼續之前，您需要設定一個 Firebase 專案：
1.  如果您尚未建立 Firebase 項目，請在 [Firebase 控制台](https://console.firebase.google.com/) 中點選“新增項目”，然後依照螢幕上的指示建立 Firebase 專案或將 Firebase 服務新增至現有 GCP 專案。
<img src="https://i.imgur.com/B8njkTG.png" alt="Welcome to Firebase" width=50%>
2. 然後，開啟您的專案並前往 **專案設定** 頁面，建立服務帳戶並使用 **產生新的私鑰** 下載服務帳戶金鑰檔案。確保此文件安全，因為它授予管理員對您的專案的存取權限。
<img src="https://i.imgur.com/J20U7lz.png" alt="Project Overview" width=50%>
<img src="https://i.imgur.com/46FOyMm.png" alt="Service accounts" width=50%>

3. 上傳 JSON 服務帳戶金鑰檔案並將其位置設定在 `GOOGLE_APPLICATION_CREDENTIALS` 環境變數中。


In [ ]:
import os
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

  with open('/content/' + fn, 'wb') as f:
    f.write(uploaded[fn])

  os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/content/' + fn

這將允許您向 Firebase 進行身份驗證並使用其服務。

In [ ]:
# Get the project ID
import json

with open(os.environ["GOOGLE_APPLICATION_CREDENTIALS"], 'r') as f:
    data = json.load(f)
    PROJECT_ID = data['project_id']
    print(PROJECT_ID)

my-genkit-gemma-firebase-demo


## 建立 Cloud Firestore 資料庫

現在您已經設定了 Firebase 和開發環境，讓我們繼續建立 Firestore 資料庫。
* 導覽至 [Firebase 控制台](https://console.firebase.google.com/project/_/firestore) 的 **Cloud Firestore** 部分。您將被prompted 選擇一個現有的 Firebase 專案。遵循資料庫建立工作流程。
<img src="https://i.imgur.com/WNmcCXa.png" alt="Service accounts" width=50% height=50%>
* 為了簡化我們的演示，我們為您的**Cloud Firestore 安全性規則**選擇一種啟動模式。您將選擇**測試模式**來快速開始。

* 選擇一個位置，此設定將是您專案的預設 Google Cloud 平台 (GCP) 資源位置。

**注意**：測試模式允許對資料庫進行開放訪問，這對於生產環境來說是不安全的。請記住在部署應用程式之前更新您的安全規則。

## Create a vector index for Firestore

在使用向量嵌入執行最近鄰搜尋之前，必須建立相應的索引。

為此，我們首先驗證 Google Cloud SDK 以簡化其創建。

In [ ]:
!gcloud auth login --no-launch-browser

請依照瀏覽器中的驗證流程進行操作，並在prompted 時將驗證程式碼複製回Colab@notebook。

Firestore 依賴索引來提供快速且有效率的集合查詢。
**注意**：這裡的「索引」指的是資料庫索引，而不是 Genkit 的索引器和檢索器抽象。
本教學要求嵌入欄位被索引才能運作。
請依照 [Firestore 文件](https://firebase.google.com/docs/firestore/vector-search?authuser=0#create_and_manage_vector_indexes) 中的說明執行以下 `gcloud` 指令來建立單一欄位向量索引。
* `collection-group` 是採集組的 ID。
* `vector-field` 是包含向量嵌入的欄位的名稱。
* `field-config` 包含向量設定（向量維度和索引類型）。維度是最大2048的整數。索引類型必須是扁平的。您也可以在此指定`field-path`，即`embedding`。

In [ ]:
%%bash -s "$PROJECT_ID"

# Set the current project ID
gcloud config set project $1

# Create a vector index
gcloud alpha firestore indexes composite create \
  --project=$1 \
  --collection-group=merch \
  --query-scope=COLLECTION \
  --field-config=vector-config='{"dimension":"768","flat": "{}"}',field-path=embedding

## 檢索增強生成 (RAG)

**Firebase Genkit** 提供可協助您建立檢索增強產生 (RAG) 流的抽象，以及提供與相關工具整合的插件。
RAG是什麼？
檢索增強生成是一種用於將外部資訊來源納入法學碩士回答的技術。能夠做到這一點很重要，因為雖然法學碩士通常接受廣泛材料的培訓，但法學碩士的實際使用通常需要特定的領域知識（例如，您可能希望使用法學碩士來回答客戶有關您公司產品的問題）。
核心 Genkit framework 提供抽象來幫助您完成 RAG：
* **索引器**：將文件新增至`"index"`。
* **嵌入器**：將文件轉換為向量表示
* **檢索器**：根據查詢從 `"index"` 檢索文件。
這些定義故意很廣泛，因為 Genkit 對於 `"index"` 是什麼或如何從中檢索文件沒有明確的看法。 Genkit 僅提供 `Document` 格式，其他所有內容均由檢索器或索引器實作提供者定義。

您很快就會了解如何將產品描述集合提取到向量資料庫中，並檢索它們以在確定可用項目的流程中使用。您甚至可以詢問有關自訂資料的一般問題，Gemma 應該能夠理解從使用者查詢檢索到的相關上下文。

###  Genkit 項目設置

建立一個新的專案目錄並初始化一個 NPM 專案。

In [ ]:
# Create project directory
!mkdir genkit-gemma-sample
%cd genkit-gemma-sample

# Initialize NPM project
!npm init -y

/content/genkit-gemma-sample
Wrote to /content/genkit-gemma-sample/package.json:

{
  "name": "genkit-gemma-sample",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}



⠙

初始化 Genkit 專案並建立一個使用舊 Gemma 模型的範例 Ollama 專案。您稍後將更新它以使用最新模型。

In [ ]:
!genkit init --model ollama --non-interactive

### 為RAG準備數據

建立一個名為`products.txt` 的文件，其中包含您希望chatbot 了解的產品的描述。

In [ ]:
%%writefile products.txt

**Google Pixel 8 Pro - Obsidian - Google Store**

The Google Pixel 8 Pro is Google's latest flagship smartphone, featuring a 6.7-inch LTPO OLED display with a 120Hz refresh rate for smooth and vibrant visuals.
Powered by the Google Tensor G3 chip, it offers exceptional performance, advanced AI capabilities, and enhanced security with the Titan M2 security chip.
The Pixel 8 Pro boasts a versatile triple-camera system, including a 50 MP main sensor, a 48 MP telephoto lens, and a 48 MP ultra-wide lens, enabling you to capture high-quality photos and videos in various lighting conditions.
Innovative camera features like Magic Eraser, Night Sight, and Super Res Zoom enhance your photography experience. The device supports 5G connectivity, has an all-day battery life with fast charging and wireless charging capabilities, and runs on the latest Android OS with guaranteed software updates.

* **Price:** Starting at $999
* **Reviews:** 4.8 out of 5 stars based on customer reviews on the Google Store

---

**Samsung Galaxy Watch5 Pro - 45mm Bluetooth Smartwatch - Black Titanium - Samsung**

The Samsung Galaxy Watch5 Pro is a premium smartwatch designed for outdoor enthusiasts and fitness aficionados.
Featuring a durable Titanium case and Sapphire Crystal Glass, it's built to withstand tough conditions.
The watch includes advanced health monitoring features like ECG, blood pressure measurement, and body composition analysis. It offers GPS route tracking, turn-by-turn navigation, and has a battery life of up to 80 hours.
The Galaxy Watch5 Pro runs on Wear OS Powered by Samsung, providing access to a wide range of apps.

* **Price:** $449.99
* **Reviews:** 4.5 out of 5 stars based on customer reviews on the Samsung website

---

**Dell XPS 13 Laptop - 13.4-inch FHD+ Display - Platinum Silver - Dell**

The Dell XPS 13 is a compact and powerful laptop featuring a 13.4-inch InfinityEdge FHD+ display.
Powered by up to the 11th Gen Intel Core processors, it delivers excellent performance for multitasking and creative work.
The laptop boasts a sleek design with a machined aluminum chassis and a carbon fiber palm rest.
It includes up to 16 GB of RAM and up to 1 TB of SSD storage. With a long battery life and Wi-Fi 6 connectivity, the XPS 13 is ideal for professionals on the go.

* **Price:** Starting at $999.99
* **Reviews:** 4.6 out of 5 stars based on customer reviews on Dell's website

---

**Bose QuietComfort 45 Wireless Noise-Cancelling Headphones - Black - Bose**

The Bose QuietComfort 45 headphones offer world-class noise cancellation with two modes: Quiet and Aware.
They deliver high-fidelity audio with a balanced performance at any volume. The headphones are lightweight and feature synthetic leather ear cushions for all-day comfort.
With up to 24 hours of battery life on a single charge, they are perfect for long flights or extended listening sessions. The headphones support Bluetooth 5.1 for a strong and reliable wireless connection.

* **Price:** $329
* **Reviews:** 4.8 out of 5 stars based on customer reviews on Bose's website

---

**Canon EOS R6 Mirrorless Camera Body - Canon Online Store**

The Canon EOS R6 is a full-frame mirrorless camera designed for both enthusiasts and professionals.
It features a 20.1 MP CMOS sensor and the DIGIC X image processor, providing excellent image quality and high-speed performance.
The camera offers up to 12 fps mechanical shutter and 20 fps electronic (silent) shutter, making it ideal for action photography.
It includes 4K UHD video recording, in-body image stabilization, and Dual Pixel CMOS AF II for fast and accurate autofocus.
The EOS R6 has built-in Wi-Fi and Bluetooth for easy sharing and remote control.

* **Price:** $2,499
* **Reviews:** 4.7 out of 5 stars based on customer reviews on the Canon Online Store

---

**Apple AirPods Pro (2nd Generation) - White - Apple Store**

The Apple AirPods Pro (2nd Generation) offer superior sound quality with Active Noise Cancellation and Adaptive Transparency.
Equipped with the H2 chip, they deliver high-fidelity audio with personalized Spatial Audio features.
The earbuds come with four sizes of silicone ear tips for a customizable fit and include touch controls for media playback and volume adjustment.
With improved battery life, you get up to 6 hours of listening time on a single charge and up to 30 hours with the MagSafe Charging Case.

* **Price:** $249
* **Reviews:** 4.7 out of 5 stars based on customer reviews on the Apple Store

---


Writing products.txt


### 使用 **Dotprompt** 建立提示文件

Firebase Genkit 提供了 Dotprompt 外掛程式和文字格式來幫助您編寫和組織生成式 AI prompt。
Dotprompt 幫助您組織和管理語言模型使用的prompt。建立 .prompt 檔案來定義模型應如何與資料和使用者互動。這使得 prompt 的維護和版本化變得更加容易，類似於管理程式碼的方式。

在`src/prompts`目錄中建立`assistant.prompt`。

In [ ]:
# Create a `prompts` directory to store your Dotprompts
!mkdir -p prompts

In [ ]:
%%writefile prompts/assistant.prompt
---
model: ollama/gemma2:2b
config:
  temperature: 0.8
input:
  schema:
    data(array): string
    question: string
output:
  format: text
---
You are acting as a helpful AI assistant that can answer questions using the data that's available.

Use only the context provided to answer the question.
If you don't know, do not make up an answer.

Context:
{{#each data~}}
- {{this}}
{{~/each}}

Question:
{{question}}

Writing prompts/assistant.prompt


### 分塊、嵌入和索引

然後，您將使用 Genkit 將這些產品描述嵌入並索引到 Firestore 中，以便 chatbot 可以透過執行以下步驟在回答問題時檢索它們：
* **分塊**：接下來，使用 `llm-chunk` 將這些產品描述分成更小的、可管理的區塊。對資料進行分塊有助於確保內容的大小適合嵌入，使其在使用向量表示時更加有效。 `llm-chunk` library 提供了將文字拆分為可向量化片段的簡單方法。

* **嵌入**：嵌入器是一種獲取內容（文字、圖像、音訊等）並創建對原始內容的語義進行編碼的數位向量的函數。若要填入您的 Firestore 集合，請使用 Google AI 中的 `Gecko embeddings` 以及 `Firebase Admin SDK`。

* **索引**：建立嵌入後，將它們索引到 Firestore 中，以便稍後用於相似性搜尋。將文字及其嵌入儲存在 Firestore 中。

在`src/flows`目錄中建立`embedFlow.ts`。

In [ ]:
%%writefile src/embedFlow.ts

import { configureGenkit } from "@genkit-ai/core";
import { embed } from "@genkit-ai/ai/embedder";
import { defineFlow, run } from "@genkit-ai/flow";
import { textEmbeddingGecko001, googleAI } from "@genkit-ai/googleai";
import { FieldValue, getFirestore } from "firebase-admin/firestore";
import { chunk } from "llm-chunk";
import * as z from "zod";
import { readFile } from "fs/promises";
import path from "path";

// Configuration for indexing process
const indexConfig = {
  collection: "merch",  // Firestore collection to store the data
  contentField: "text", // Field name for the text content
  vectorField: "embedding", // Field name for the embedding vector
  embedder: textEmbeddingGecko001, // Embedder model to use
};

// Configure Genkit with Google AI plugin
// Firebase Genkit has a configuration and plugin system.
// Every Genkit app starts with configuration where you specify the plugins
// you want to use and configure various subsystems.
configureGenkit({
  plugins: [googleAI({ apiVersion: ['v1', 'v1beta'] })],
  enableTracingAndMetrics: false,
});

// Initialize Firestore instance
const firestore = getFirestore();

// Create chunking config
// This example uses the llm-chunk library which provides a simple text
// splitter to break up documents into segments that can be vectorized.
const chunkingConfig = {
  minLength: 1000,
  maxLength: 2000,
  splitter: 'sentence',
  overlap: 100,
  //  Split text into chunks using '---' as delimiter
  delimiters: '---',
} as any;

// Define embed flow
export const embedFlow = defineFlow(
  {
    name: "embedFlow", // Name of the flow
    inputSchema: z.void(), // No input is expected
    outputSchema: z.void(), // No output is returned
  },
  async () => {
    // Read text data from file
    const filePath = path.resolve('products.txt');
    const textData = await run("extract-text", () => extractText(filePath));

    // Divide the text into segments.
    const chunks = await run('chunk-it', async () =>
      chunk(textData, chunkingConfig)
    );

    // Index chunks into Firestore.
    await run("index-chunks", async () => indexToFirestore(chunks));
  }
);

// Function to index chunks into Firestore
async function indexToFirestore(data: string[]) {
  for (const text of data) {
    // Generate embedding for the text chunk
    const embedding = await embed({
      embedder: indexConfig.embedder,
      content: text,
    });

    // Add the text and embedding to Firestore
    await firestore.collection(indexConfig.collection).add({
      [indexConfig.vectorField]: FieldValue.vector(embedding),
      [indexConfig.contentField]: text,
    });
  }
}

// Function to read text content from a file
async function extractText(filePath: string) {
  const f = path.resolve(filePath);
  return await readFile(f, 'utf-8');
}

Writing src/embedFlow.ts


透過儲存文字及其嵌入，您可以稍後執行相似性搜索，以根據使用者查詢尋找相關產品描述。這使得 chatbot 能夠檢索並提供準確的、上下文相關的答案。

### 設定和插件

Firebase Genkit 有一個設定和插件系統。每個 Genkit 應用程式都從設定開始，您可以在其中指定要使用的插件並設定各種子系統。
在`src`目錄中建立`config.ts`。

In [ ]:
%%writefile src/config.ts

import { configureGenkit } from '@genkit-ai/core';
import { firebase } from '@genkit-ai/firebase';
import { googleAI } from '@genkit-ai/googleai';
import { ollama } from 'genkitx-ollama';
import { dotprompt } from '@genkit-ai/dotprompt';
import { initializeApp, applicationDefault } from 'firebase-admin/app';
import { getFirestore } from 'firebase-admin/firestore';

// Initialize Firebase Admin SDK
const app = initializeApp({
  credential: applicationDefault(),
});

export const firestore = getFirestore(app);

// Configure Genkit
configureGenkit({
  plugins: [
    firebase(),
    googleAI({ apiVersion: ['v1', 'v1beta'] }),
    ollama({
      // Ollama provides an interface to many generative models. Here,
      // you specify Google's Gemma 2 model. The models you specify must already
      // be downloaded and available to the Ollama server.
      models: [{ name: 'gemma2:2b' }],
      // The address of your Ollama API server. This is often a different host
      // from your app backend (which runs Genkit), in order to run Ollama on
      // a GPU-accelerated machine.
      serverAddress: 'http://127.0.0.1:11434',
    }),
    dotprompt(),
  ],
  // Log debug output to tbe console.
  logLevel: 'debug',
  // Perform OpenTelemetry instrumentation and enable trace collection.
  enableTracingAndMetrics: true,
});

Writing src/config.ts


### 定義 RAG 流

接下來，建立一個名為 `chatbotFlow` 的串流，該串流將允許 chatbot 與您先前索引的資料進行互動。此流程將檢索器（從 Firebase 取得相關資訊）與協助格式化回應的prompt 結合。檢索器是封裝與任何類型的文件檢索相關的邏輯的概念。最受歡迎的檢索案例通常包括從向量儲存中檢索；然而，在 Genkit 中，檢索器可以是任何返回資料的函數。在這種情況下，檢索器負責根據使用者的問題從 Firestore 尋找最相關的產品描述。它使用**嵌入**和**餘弦相似度**來找到最接近的匹配，確保檢索到的信息與查詢高度相關。
在`src`目錄中建立`memory.ts`和`chatbotFlow.ts`。

In [ ]:
%%writefile src/memory.ts

import { MessageData } from '@genkit-ai/ai/model';

const chatHistory: Record<string, MessageData[]> = {};

export interface HistoryStore {
  load(id: string): Promise<MessageData[] | undefined>;
  save(id: string, history: MessageData[]): Promise<void>;
}

// You'll also use an in-memory store to store the chat history.
export function inMemoryStore(): HistoryStore {
  return {
    async load(id: string): Promise<MessageData[] | undefined> {
      return chatHistory[id];
    },
    async save(id: string, history: MessageData[]) {
      chatHistory[id] = history;
    },
  };
}

Writing src/memory.ts


In [ ]:
%%writefile src/chatbotFlow.ts

import { defineFlow, run } from '@genkit-ai/flow';
import { defineFirestoreRetriever } from '@genkit-ai/firebase';
import { retrieve } from '@genkit-ai/ai/retriever';
import { textEmbeddingGecko001 } from '@genkit-ai/googleai';
import { z } from 'zod';

import { firestore } from './config';
import { inMemoryStore } from './memory.js';

import { promptRef } from '@genkit-ai/dotprompt';

// Define Firestore retriever
const retrieverRef = defineFirestoreRetriever({
  name: "merchRetriever",
  firestore,
  collection: "merch",  // Collection containing merchandise data
  contentField: "text",  // Field for product descriptions
  vectorField: "embedding", // Field for embeddings
  embedder: textEmbeddingGecko001, // Embedding model
  distanceMeasure: "COSINE", // Similarity metric
});

// Define the prompt reference
const assistantPrompt = promptRef('assistant');

// To store the chat history
const historyStore = inMemoryStore();

// Define chatbot flow
export const chatbotFlow = defineFlow(
  {
    name: "chatbotFlow",
    inputSchema: z.string(),
    outputSchema: z.string(),
  },
  async (question) => {
    const conversationId = '0';

    // Retrieve conversation history.
    const history = await run(
      'retrieve-history',
      conversationId,
      async () => {
        return (await historyStore?.load(conversationId)) || [];
      }
    );

    // Retrieve relevant documents
    const docs = await retrieve({
      retriever: retrieverRef,
      query: question,
      options: { limit: 5 },
    });

    // Run the prompt
    const mainResp = await assistantPrompt.generate({
      history: history,
      input: {
        data: docs.map((doc) => doc.content[0].text || ""),
        question: question,
      },
    });

    // Save history.
    await run(
      'save-history',
      {
        conversationId: conversationId,
        history: mainResp.toHistory(),
      },
      async () => {
        await historyStore?.save(conversationId, mainResp.toHistory());
      }
    );

    // Handle the response from the model API. In this sample, we just convert
    // it to a string, but more complicated flows might coerce the response into
    // structured output or chain the response into another LLM call, etc.
    return mainResp.text();
  }
);

Writing src/chatbotFlow.ts


檢索器與 LLM（Gemma 2 透過 Ollama）合作建立檢索增強產生 (RAG) 流程。檢索器會取得相關文檔，然後語言模型使用這些文檔產生準確的回應，將常識與自訂資料中的特定相關資訊結合。
chatbot流程由幾個關鍵步驟組成：
* **Firestore Retriever**：`retrieverRef` 指定如何從 Firestore 取得數據，使用 `contentField`（產品說明）和 `vectorField`（嵌入）等欄位來尋找相關資訊。

* **提示參考**：`assistantPrompt` 引用您先前使用 Dotprompt 建立的 prompt，確定助手應如何格式化回應。

* **檢索並產生回應**：chatbot 流程檢索相關文件並將其用作上下文來產生回應。它利用歷史背景來提供連貫且與背景相關的答案。

最後，透過在 `src/index.ts` 中定義 chatbotFlow 和 embedFlow 來封裝 Genkit 應用程式. 此腳本啟動一個串流伺服器，將您的串流公開為 HTTP 端點，讓您可以與您定義的串流互動：

In [ ]:
%%writefile src/index.ts

import { startFlowsServer } from '@genkit-ai/flow';
import { chatbotFlow } from './chatbotFlow';
import { embedFlow } from './embedFlow';

// Start a flow server, which exposes your flows as HTTP endpoints. This call
// must come last, after all of your plug-in configuration and flow definitions.
// You can optionally specify a subset of flows to serve, and configure some
// HTTP server options, but by default, the flow server serves all defined flows.
startFlowsServer({
  flows: [chatbotFlow, embedFlow],
});

Overwriting src/index.ts


### 啟動 Genkit 伺服器

自動按`Enter`或`\n`接受以下條款。

> Genkit CLI 和開發人員 UI 使用 Google 的 cookie 和類似技術來提供和提高其服務品質並分析使用情況。了解更多https://policies.google.com/technologies/cookies

In [ ]:
import os
from google.colab import userdata

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

command = [
    "genkit", "start", "-o", "--port", "8081"
]

# Create a file to write logs
with open("genkit.log", "w") as logfile:
  # Use subprocess.Popen to run the command with nohup-like behavior
  genkit_process = subprocess.Popen(
    command,
    stdout=logfile,
    stderr=subprocess.STDOUT,
    stdin=subprocess.PIPE,
    start_new_session=True  # This is similar to nohup behavior, detaches from terminal
  )
  # Send an Enter key (\n) to the process to accept the terms
  genkit_process.stdin.write(b'\n')
  genkit_process.stdin.flush()

# Sleep for 60 seconds
time.sleep(60)

## 公開 Genkit 工具 Web API

使用Colab的代理公開伺服器的工具API端點。如果您需要偵錯任何問題，可以透過這種方式存取 Web 介面。

In [ ]:
# Uncomment the following code to access the web interface

# from google.colab.output import eval_js
# proxy_url = eval_js("google.colab.kernel.proxyPort(8081)")

# print(f"The Genkit Tools UI is accessible at: {proxy_url}")

## 使用`embedFlow`來索引文檔

現在，使用 HTTP POST curl 請求執行 `embedFlow` 以索引 `.txt` 文件內的文檔

In [ ]:
!curl -X POST "http://127.0.0.1:3400/embedFlow" \
  -H "Content-Type: application/json" \
  -d '{}'

您應該看到一條訊息，指示文件已成功建立索引。

## 使用 `chatbotFlow` 嘗試 RAG

最後，您可以查詢RAG chatbot來詢問一些有關您的資料的簡單問題。

In [ ]:
!curl -X POST "http://127.0.0.1:3400/chatbotFlow" \
  -H "Content-Type: application/json" \
  -d '{"data": "What is the price of the Pixel 8 Pro?"}'

{"result":"The price of the Pixel 8 Pro starts at $999. \n"}

## （可選）使用 Gradio 聊天機器人介面進行聊天

使用 **Gradio** 建立一個簡單的 Web 介面。

In [ ]:
import gradio as gr
import requests


def chat(question, history):
    try:
        response = requests.post(
            "http://127.0.0.1:3400/chatbotFlow",
            headers={"Content-Type": "application/json"},
            json={"data": question}
        )

        # Check for HTTP request errors
        response.raise_for_status()

        json_response = response.json()

        if 'result' in json_response:
            return json_response['result']
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return "Sorry, an unexpected error occurred."


gr.ChatInterface(
  chat,
  chatbot=gr.Chatbot(
    show_copy_button=True,
    elem_id="chatbot",
    render=False,
    render_markdown=True,
    height=300
  ),
  textbox=gr.Textbox(placeholder="Ask me a question", container=False, scale=7),
  title="Firebase Genkit RAG Chatbot",
  description="Ask any question about products.",
  theme="soft",
  examples=[
      {"text": "What is the price of the Pixel 8 Pro?"},
      {"text": "Tell me about the battery life of the Samsung Galaxy Watch5 Pro."},
      {"text": "Does the Google Pixel 8 Pro support 5G connectivity?"}
  ],
  show_progress=True
).launch(debug=True)

這將產生一個公共 URL，您可以使用它來存取 chatbot 介面。

現在您的文件已建立索引並且 Gradio 介面正在執行，您可以開始與 RAG 應用程式進行互動。
使用提供的 URL 開啟Gradio 介面並詢問有關資料的問題，例如：
- “Pixel 8 Pro 的價格是多少？”
- “告訴我三星 Galaxy Watch5 Pro 的電池續航力。”
- “Google Pixel 8 Pro 支援 5G 連線嗎？”

您應該會收到基於 `.txt` 文件中提供的數據的答案。

## 清理

當您接近本教學的結尾時，讓我們清理所有內容。

In [ ]:
# Terminate all processes
ollama_serve_process.terminate()
ollama_run_process.terminate()
genkit_process.terminate()

# Delete Firebase project (press Y to confirm)
!gcloud projects delete "$PROJECT_ID"

恭喜！您已使用 **Genkit**、**Firebase**、**Ollama**、**Gemma**、**Dotprompt** 和 **Gradio@P0005@@**、**Dotprompt** 和 **Gradio** 成功建立了 RAG